# OPERA — large-model run on Colab (A100)

Attention-free byte-level OPERA with the gated quaternion holographic memory, trained on
FineWeb-Edu bytes. Plan and results: `docs/OPERA_Recall_Research_2026-09-24.md` §8b–§8c.

**Order (run top to bottom; every step is idempotent and caches to Drive):**
1. Setup: GPU check, Drive, code, dependencies.
2. Correctness on this GPU: the Triton kernel test and the CUDA-vs-CPU model check. **Stop if either fails.**
3. Data: the Simple-Wikipedia test sets (for continuity with every earlier run), then the FineWeb-Edu pool.
4. Throughput benchmark of every ladder rung → the ladder plan (steps and GPU-hours per rung).
5. The ladder: rungs train one after another; re-running the cell after a disconnect **resumes** (finished rungs are skipped, the running one continues from its last checkpoint on Drive).
6. The fit: loss against compute, with the depth check.

**Demo model (sections 7–11):** after a reconnect, run sections 1–3 again (fast; everything is cached on Drive),
skip 4–6, then run 7 → 11: the 18 h pretraining run, the chat data, chat fine-tuning, export to the Hub,
and the Space. Each step resumes or skips when re-run.

Runtime → Change runtime type → **A100 GPU** (High-RAM on).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), 'no GPU: Runtime -> Change runtime type -> A100'
print(torch.__version__, torch.cuda.get_device_name(), 'bf16 native:', torch.cuda.get_device_capability()[0] >= 8)

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/opera_scale'      # everything that must survive a disconnect
RUNS = f'{DRIVE}/runs'                             # checkpoints, summaries, logs
DDATA = f'{DRIVE}/data'                            # cached datasets
LOCAL = '/content/data'                            # fast local copy used for training
REPO = '/content/opera-lm'
for d in (RUNS, DDATA, LOCAL):
    os.makedirs(d, exist_ok=True)
os.environ['OPERA_RUNS'] = RUNS

In [ ]:
# Code: the main branch. For a PRIVATE repo add a Colab secret GITHUB_TOKEN
# (key icon in the left bar; a fine-grained token with read access to OPERA-LM).
BRANCH = 'main'
try:
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN')
except Exception:
    tok = None
url = (f'https://{tok}@github.com/Merna-Khalid/OPERA-LM.git' if tok
       else 'https://github.com/Merna-Khalid/OPERA-LM.git')
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} {url} {REPO}
else:
    !git -C {REPO} pull -q
!git -C {REPO} log -1 --oneline
%cd {REPO}
!pip -q install datasets

## 2. Correctness on this GPU (stop if anything fails)

In [ ]:
!python -m opera_lm.triton_kernel --test
!python experiments/device_check.py --device cuda
!python experiments/device_check.py --device cuda --layers 4

## 3. Data

**3a. Simple-Wikipedia test sets** — the a20000 corpus (its test set is what every earlier BPB was
measured on) and the full-corpus pool test set. Rebuilt from the fixed `20231101.simple` snapshot and
checked against the counts measured locally; cached on Drive.

In [ ]:
import json, pickle, shutil
need = ['corpus_bytes_T1024_a20000.pkl', 'packed_bytes_T1024_all.test.pkl']
for f in need:
    if os.path.exists(f'{DDATA}/{f}') and not os.path.exists(f'assets/{f}'):
        shutil.copy(f'{DDATA}/{f}', f'assets/{f}')
if not os.path.exists('assets/corpus_bytes_T1024_a20000.pkl'):
    !python experiments/build_reprs.py --max-articles 20000 --modes bytes
if not os.path.exists('assets/packed_bytes_T1024_all.test.pkl'):
    !python experiments/build_packed.py
(tr, ts, tl, V), meta = pickle.load(open('assets/corpus_bytes_T1024_a20000.pkl', 'rb'))
assert (len(tr), len(ts), len(tl), sum(map(len, tr))) == (45765, 5092, 252, 37548997), 'a20000 corpus differs from the local build'
pm = json.load(open('assets/packed_bytes_T1024_all.meta.json')) if os.path.exists('assets/packed_bytes_T1024_all.meta.json') else None
if pm:
    assert (pm['n_seqs'], pm['total_tokens']) == (338339, 216919731), 'pool differs from the local build'
pt = pickle.load(open('assets/packed_bytes_T1024_all.test.pkl', 'rb'))['meta']
assert (pt['n_short_all'], pt['n_long']) == (37817, 1035), 'pool test set differs from the local build'
for f in need:
    if not os.path.exists(f'{DDATA}/{f}'):
        shutil.copy(f'assets/{f}', f'{DDATA}/{f}')
print('Simple-Wikipedia test sets OK')

**3b. FineWeb-Edu byte pool.** `MAX_BYTES` = training bytes in the pool. 4 GB covers the default
ladder: its largest rung (77M params × 20) needs 1.5 GB, so every rung sees under one pass. Built once
(streams from the Hub, ~20–60 min for 4 GB), cached on Drive, copied to local disk for training.

In [ ]:
MAX_BYTES = 4e9
tag = f'fineweb_bytes_T1024_{MAX_BYTES / 1e9:g}GB'
FW = f'{LOCAL}/{tag}'
files = [f'{tag}.{s}' for s in ('tokens.npy', 'offsets.npy', 'test.pkl', 'meta.json')]

def sync(names):
    """Copy Drive -> local disk, re-copying any local file whose size differs
    (a half-copied file from an interrupted session is never trusted)."""
    for f in names:
        src_, dst = f'{DDATA}/{f}', f'{LOCAL}/{f}'
        if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src_):
            print('copying', f, f'({os.path.getsize(src_) / 1e9:.2f} GB)', flush=True)
            shutil.copy(src_, dst)
        assert os.path.getsize(dst) == os.path.getsize(src_), f'incomplete copy: {f}'

if all(os.path.exists(f'{DDATA}/{f}') for f in files):
    sync(files)
else:
    !python experiments/build_fineweb_bytes.py --max-bytes {MAX_BYTES} --out-dir {LOCAL}
    for f in files:
        shutil.copy(f'{LOCAL}/{f}', f'{DDATA}/{f}')
import numpy as np
tok = np.load(f'{FW}.tokens.npy', mmap_mode='r'); off = np.load(f'{FW}.offsets.npy', mmap_mode='r')
assert off[-1] == tok.size, 'pool is corrupt: offsets do not match tokens'
print(f'pool OK: {tok.size / 1e9:.2f}G bytes, {off.size - 1:,} sequences')
!df -h /content | tail -1

## 4. Throughput benchmark and ladder plan

One training step (forward + backward + Muon/AdamW, bf16, torch.compile) for every ladder rung at
batch 32. The first call of each size includes compilation. `ladder.py plan` turns the measurements
into steps and GPU-hours per rung (each rung trains on `RATIO` = 20 bytes per parameter).
**Check the total before starting the ladder**; drop the largest rung (`2048x2`) from `RUNGS` if it
does not fit the budget.

In [ ]:
RUNGS = '512x2 768x2 1024x2 1536x2 2048x2 1024x4'   # width ladder at L=2 + depth check (d1024 x L4 ~ d1536 x L2)
RATIO, BATCH = 20, 32
grid = ' '.join(f'{r}x{BATCH}' for r in RUNGS.split())
!python experiments/bench_scale.py --out {RUNS}/bench.jsonl --grid {grid}
!python experiments/ladder.py plan --pool {FW} --rungs {RUNGS} --ratio {RATIO} --batch {BATCH}

## 5. The ladder

Trains every rung in order (`fw_d{width}_L{layers}_r{RATIO}`), each scored at the end on the
FineWeb-Edu held-out set and the Simple-Wikipedia sets. Logs: `runs/<arm>.log` on Drive.
**After a disconnect, just re-run this cell**: finished rungs are skipped and the interrupted one
resumes its exact batch stream from the last checkpoint (every 1000 steps).

In [ ]:
!python experiments/ladder.py run --pool {FW} --rungs {RUNGS} --ratio {RATIO} --batch {BATCH} --device cuda

## 6. The fit

BPB on FineWeb-Edu against training compute C = 6·params·bytes, fitted as BPB = E + A·C^(−α) over the
L=2 width ladder, with the depth check compared to the fit at equal compute. Paste the printed table
(and `runs/ladder_fit.png`) back into the conversation.

In [ ]:
!python experiments/ladder.py fit
from IPython.display import Image
Image(f'{RUNS}/ladder_fit.png')

## 7. The long run — the demo model (~18 h)

d3072 × 2 layers (~172M parameters), trained for `LONG_HOURS` on FineWeb-Edu: about 0.22 GB of text per
hour, so 18 h ≈ the whole 4 GB pool once. A one-time ~3 min measurement of this size's step time makes
the step count match the hours. **Re-run this cell after a disconnect**: it resumes from the last
checkpoint (every 1000 steps) on Drive.

In [ ]:
import sys, argparse
LONG_RUNG, LONG_HOURS = '3072x2', 18
measured = any(r['d'] == 3072 and r['L'] == 2 and r['B'] == 32
               for r in map(json.loads, open(f'{RUNS}/bench.jsonl')))
if not measured:
    !python experiments/bench_scale.py --out {RUNS}/bench.jsonl --grid 3072x2x32
sys.path.insert(0, 'experiments')
import ladder
ARM = ladder.plan(argparse.Namespace(pool=FW, rungs=[LONG_RUNG], ratio=20, hours=LONG_HOURS,
                                     batch=32, bench=f'{RUNS}/bench.jsonl'))[0]['arm']
print('long-run arm:', ARM)
!python experiments/ladder.py run --pool {FW} --rungs {LONG_RUNG} --hours {LONG_HOURS} --batch 32 --device cuda

## 8. Chat data — smol-smoltalk as bytes

Downloads HuggingFaceTB/smol-smoltalk (on Colab's disk, not your computer) and formats every
conversation in the byte chat format (`opera_lm/chat.py`), up to 2,048 bytes, cut at turn boundaries.
Cached on Drive.

In [ ]:
CHAT = f'{LOCAL}/chat_bytes_T2048'
cfiles = [f'chat_bytes_T2048.{s}' for s in ('tokens.npy', 'offsets.npy', 'test.pkl', 'meta.json')]
if all(os.path.exists(f'{DDATA}/{f}') for f in cfiles):
    sync(cfiles)
else:
    !python experiments/build_chat_bytes.py --out-dir {LOCAL}
    for f in cfiles:
        shutil.copy(f'{LOCAL}/{f}', f'{DDATA}/{f}')
print(json.load(open(f'{CHAT}.meta.json'))['splits'])

## 9. Chat fine-tuning (~1 h)

Starts from the long run's weights, lower learning rate, conversations up to 2,048 bytes. Prints the
held-out chat BPB and three sample replies at the end. Re-run to resume.

In [ ]:
!python experiments/sft.py --arm {ARM} --pool {CHAT} --hours 1.0

## 10. Export to the Hugging Face Hub

Add a Colab secret **HF_TOKEN** (a Hugging Face token with *write* access) and set `HF_REPO` to
`<your-hf-username>/opera-lm-chat`. Pushes `config.json`, `model.safetensors` and a model card.

In [ ]:
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
HF_REPO = 'YOUR_HF_USERNAME/opera-lm-chat'
assert not HF_REPO.startswith('YOUR_'), 'set HF_REPO to your Hugging Face username'
!python experiments/export_hf.py --arm {ARM} --ckpt-dir {RUNS}/{ARM}_sft --out /content/export --push {HF_REPO}

## 11. Publish the demo Space

Creates a Gradio Space (free CPU) from `demo/`, pointed at the model repo. Replies stream at roughly
8–10 characters per second on the free CPU.

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=os.environ['HF_TOKEN'])
SPACE = HF_REPO.split('/')[0] + '/opera-lm-chat-demo'
api.create_repo(SPACE, repo_type='space', space_sdk='gradio', exist_ok=True)
api.add_space_variable(SPACE, 'MODEL_REPO', HF_REPO)
api.upload_folder(folder_path='demo', repo_id=SPACE, repo_type='space')
print('Space:', f'https://huggingface.co/spaces/{SPACE}')